# Phase 4A — Milestone 5: FRED Publication-Lag Leakage Investigation

This notebook quantifies the performance impact of the **FRED publication-lag
fix** shipped in this milestone: the macro ASOF join now shifts each series'
observation dates forward by its pinned publication lag before the merge, so
bar *t* only sees values that were publicly knowable at the close of bar *t*.

**Reference documents**
- PRD: [`phase-4a-feature-and-label-redesign.prd.md`](../.claude/prds/phase-4a-feature-and-label-redesign.prd.md)
- Plan: [`phase-4a-milestone-5-fred-leakage.plan.md`](../.claude/plans/phase-4a-milestone-5-fred-leakage.plan.md)
- Concept doc: [`fred-publication-lag.md`](../docs/concepts/fred-publication-lag.md)
- Companion notebooks: [`05_phase4a_regime_harness.ipynb`](05_phase4a_regime_harness.ipynb), [`06_phase4a_label_ablation.ipynb`](06_phase4a_label_ablation.ipynb)

**Why this notebook exists.** nb03's in-sample SHAP rankings are dominated by
macro features (DFF, yield_curve, DGS10, VIXCLS) while the same model shows no
OOS edge — the classic signature of a look-ahead artifact. The Milestone 5
audit confirmed a *mechanical* leak: the join merged on FRED **observation
dates**, but DFF/DGS10 for day *t* are published the *next* business day, and
the official VIX close is disseminated ~4:15pm ET — after the 4:00pm close
where our signal forms. The corrected (lagged) join is already the
`build_features` default **on correctness grounds**; this notebook measures
how much the leak actually mattered on a fast slice.

**Pre-committed materiality thresholds** (pinned in the plan *before* any
result was computed — do not adjust):

| Threshold | Value |
|---|---|
| Sign-flip fraction of OOS bars | > 5% |
| &#124;ΔSharpe&#124; aggregate **or** in any era regime | > 0.1 |

The leak is "material" if **either** threshold trips. The lagged join stays
default **regardless** of the outcome.

**Pre-committed interpretation rule for the macro-only probe:** if the
unlagged macro-only model shows IS skill that materially degrades under the
1-day lag, nb03's IS macro dominance is at least partly mechanical look-ahead;
if both arms are statistically indistinguishable, join timing is not the
explanation and the report says "no leak found via publication timing."

**Sections**
1. Setup and imports
2. The leak, demonstrated
3. Macro-only leakage probe (IS + OOS, unlagged vs lagged)
4. Full-feature A/B (aggregate + per-regime Sharpe, SHAP stability, sign-flip)
5. Verdict + what's next

---
## 1 — Setup and imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print(f'numpy   {np.__version__}')
print(f'pandas  {pd.__version__}')

In [ ]:
# Milestone 5 surface area — everything we will exercise in this notebook.
from quant.features.engineering import FRED_PUBLICATION_LAGS, build_features
from quant.models.gbm import GBMModel
from quant.backtest.harness import BacktestResult, run_portfolio_backtest
from quant.backtest.regimes import DateRangeDetector, tag_regimes
from quant.backtest.regime_metrics import compute_regime_metrics
from quant.backtest.report import regime_summary_table
from quant.backtest.statistics import diebold_mariano

import xgboost as xgb

print('Milestone 5 surface area imported.')
print(f'FRED_PUBLICATION_LAGS (pinned): {FRED_PUBLICATION_LAGS}')

### Real-data slice (5 symbols × ~8 years)

Same slice convention as nb05 §6 / nb06 §4 — sized for fast iteration, not for
the final Phase 4A gate. The full-panel measurement with corrected joins
belongs to Milestone 6.

In [ ]:
import quant.storage.catalog as catalog

DEMO_SYMBOLS = ['AAPL', 'MSFT', 'JPM', 'JNJ', 'SPY']
DEMO_START = '2018-01-02'
DEMO_END = '2026-04-21'

REAL_OHLCV_LOADED = False
prices_by_sym: dict[str, pd.DataFrame] = {}
try:
    syms_sql = ', '.join(f"'{s}'" for s in DEMO_SYMBOLS)
    eq = catalog.query(f"""
        SELECT symbol, timestamp, open, high, low, close, adjClose, volume
        FROM {catalog.table('equity_eod_tiingo')}
        WHERE symbol IN ({syms_sql})
          AND timestamp BETWEEN '{DEMO_START}' AND '{DEMO_END}'
        ORDER BY symbol, timestamp
    """)
    eq['timestamp'] = pd.to_datetime(eq['timestamp']).dt.tz_localize(None)
    eq = eq.set_index('timestamp')
    for s in DEMO_SYMBOLS:
        sdf = eq[eq['symbol'] == s][['open', 'high', 'low', 'close', 'volume']].copy()
        sdf = sdf[~sdf.index.duplicated(keep='last')].sort_index()
        if not sdf.empty:
            prices_by_sym[s] = sdf
    REAL_OHLCV_LOADED = bool(prices_by_sym)
    print(f'Loaded OHLCV for {len(prices_by_sym)} symbols')
    for s, df in prices_by_sym.items():
        print(f'  {s}: {len(df):,} bars  {df.index.min().date()} → {df.index.max().date()}')
except Exception as exc:
    print(f'Lake unavailable ({type(exc).__name__}): {exc}')

---
## 2 — The leak, demonstrated

The lake stores FRED **observation dates**. The legacy join
(`fred_publication_lags=None`) attaches, to bar *t*, the most recent
observation dated ≤ *t* — i.e. the day-*t* print itself. But:

| Series   | Actual availability                                          | Pinned lag |
|----------|--------------------------------------------------------------|-----------|
| `DFF`    | NY Fed publishes EFFR the **next business day** ~9:00am ET    | **1**     |
| `DGS10`  | H.15 release; FRED vintage lands the **next business day**    | **1**     |
| `VIXCLS` | Cboe disseminates the close ~4:15pm ET — **after** the 4:00pm close where our signal forms | **1**     |

The table is static here on purpose — the empirical verification (ALFRED
vintage metadata, sampled 2026-06-12) lives in
[`docs/concepts/fred-publication-lag.md`](../docs/concepts/fred-publication-lag.md);
this notebook must run offline and never calls `fredapi`.

So under the unlagged join the model saw each macro value **one business day
before it was publicly knowable** — a mechanical look-ahead, independent of
any model. The cell below makes the shift visible: build features for one
symbol twice and diff the macro columns.

> **Side-finding (fixed in this milestone):** the audit also caught a
> session-timezone artifact in `_load_fred_wide` — the old SQL
> `CAST(timestamp AS DATE)` evaluated in the *machine-local* timezone, which
> on any US-timezone machine shifted every observation date back one calendar
> day (handing bar *t* the *t+1* observation even before the publication-lag
> question). Date extraction now happens in pandas with an explicit UTC
> conversion; see the `_load_fred_wide` docstring.

In [ ]:
demo_unlag = demo_lag = None
if REAL_OHLCV_LOADED:
    spy = {'SPY': prices_by_sym['SPY']}
    demo_unlag = build_features(['SPY'], spy, fred_publication_lags=None)['SPY']
    demo_lag = build_features(['SPY'], spy)['SPY']  # pinned FRED_PUBLICATION_LAGS default
    for df in (demo_unlag, demo_lag):
        df.index = df.index.tz_localize(None)

    MACRO_COLS = ['DGS10', 'DFF', 'VIXCLS', 'yield_curve']
    PRICE_COLS = [c for c in demo_lag.columns if c not in MACRO_COLS]
    print(PRICE_COLS)

    # Price-derived features must be bit-identical — the fix touches macro only.
    # Exclude regime columns that are explicitly derived from macro inputs.
    price_only_cols = [
        c for c in PRICE_COLS
        if c not in {'vix_regime', 'curve_inverted', 'vol_regime_ratio'}
    ]
    pd.testing.assert_frame_equal(demo_unlag[price_only_cols], demo_lag[price_only_cols])
    print('✓ Price-derived features identical between arms (fix touches macro only).')

    print('\nFraction of SPY bars whose value changes under the lagged join:')
    for c in MACRO_COLS:
        frac = float((demo_unlag[c] != demo_lag[c]).mean())
        print(f'  {c:<12} {frac:6.1%}')

    # The lagged value at bar t should equal the unlagged value at the previous
    # bar (consecutive business days) — i.e. a clean 1-day shift.
    shift_match = float((demo_lag['VIXCLS'] == demo_unlag['VIXCLS'].shift(1)).iloc[1:].mean())
    print(f'\nlagged VIXCLS[t] == unlagged VIXCLS[t-1]: {shift_match:.1%} of bars')

In [ ]:
if demo_unlag is not None:
    # 10-row window around the first bar where VIXCLS differs between arms.
    first_diff = (demo_unlag['VIXCLS'] != demo_lag['VIXCLS']).idxmax()
    loc0 = max(demo_unlag.index.get_loc(first_diff) - 2, 0)
    win = demo_unlag.index[loc0:loc0 + 10]
    side_by_side = pd.DataFrame({
        'DFF_unlagged': demo_unlag.loc[win, 'DFF'],
        'DFF_lagged': demo_lag.loc[win, 'DFF'],
        'VIXCLS_unlagged': demo_unlag.loc[win, 'VIXCLS'],
        'VIXCLS_lagged': demo_lag.loc[win, 'VIXCLS'],
    })
    side_by_side['vix_changed'] = side_by_side['VIXCLS_unlagged'] != side_by_side['VIXCLS_lagged']
    display(side_by_side)
    print('Read each row as: the lagged column is what was actually knowable at')
    print('that bar\'s close; the unlagged column is what the model used to see.')

---
## 3 — Macro-only leakage probe

Train the GBM (preview mode, `n_iter=10` — same config as nb05 §10) on the
**macro columns only** (`DGS10, DFF, VIXCLS, yield_curve`), two arms:
unlagged (`fred_publication_lags=None`) vs lagged (pinned default). If the
macro features' apparent IS skill was riding on the 1-day peek, isolating
them makes the comparison maximally sensitive.

> **Pre-committed interpretation rule:** if the unlagged macro-only model
> shows IS skill that materially degrades under the 1-day lag, nb03's IS
> macro dominance is at least partly mechanical look-ahead; if both arms are
> statistically indistinguishable, join timing is not the explanation and
> the report says "no leak found via publication timing."

Both arms are built once here and reused by the full-feature A/B in §4, so
all four backtests in this notebook run on **identical rows** (same dropna
window, same fold structure) and the only difference between arms is the
join timing.

In [ ]:
HORIZON = 1
ARMS = ('unlagged', 'lagged')

arms_feats: dict[str, dict[str, pd.DataFrame]] = {a: {} for a in ARMS}
labels_by_sym: dict[str, pd.Series] = {}
gbm_prices_by_sym: dict[str, pd.DataFrame] = {}

if REAL_OHLCV_LOADED:
    feats_raw = {
        'unlagged': build_features(DEMO_SYMBOLS, prices_by_sym, fred_publication_lags=None),
        'lagged': build_features(DEMO_SYMBOLS, prices_by_sym),
    }
    for s in DEMO_SYMBOLS:
        per_arm = {}
        for arm in ARMS:
            X = feats_raw[arm][s].copy()
            # build_features returns UTC tz-aware indexes (FRED ASOF join);
            # prices are tz-naive. Strip tz so indexes intersect (nb05 §10).
            X.index = X.index.tz_localize(None)
            per_arm[arm] = X
        fwd = (prices_by_sym[s]['close'].shift(-HORIZON) / prices_by_sym[s]['close'] - 1.0).dropna()
        common = (per_arm['unlagged'].dropna().index
                  .intersection(per_arm['lagged'].dropna().index)
                  .intersection(fwd.index))
        for arm in ARMS:
            arms_feats[arm][s] = per_arm[arm].loc[common]
        labels_by_sym[s] = fwd.loc[common]
        gbm_prices_by_sym[s] = prices_by_sym[s].loc[common]

    FULL_COLS = list(arms_feats['lagged'][DEMO_SYMBOLS[0]].columns)
    n_bars = sum(len(v) for v in labels_by_sym.values())
    print(f'Built A/B panel: {n_bars:,} (sym, bar) pairs × {len(FULL_COLS)} features.')

    # Every symbol in this slice shares one calendar — §4's prediction
    # recovery depends on it, so assert rather than assume.
    ref_idx = arms_feats['lagged'][DEMO_SYMBOLS[0]].index
    assert all(arms_feats['lagged'][s].index.equals(ref_idx) for s in DEMO_SYMBOLS), \
        'symbols do not share a common calendar — §4 sign-flip derivation invalid'
    print(f'✓ All {len(DEMO_SYMBOLS)} symbols share one calendar '
          f'({len(ref_idx):,} bars, {ref_idx.min().date()} → {ref_idx.max().date()}).')

    # The arms must actually differ in macro values (else the A/B is vacuous).
    arms_differ = any(
        (arms_feats['unlagged'][s][c] != arms_feats['lagged'][s][c]).any()
        for s in DEMO_SYMBOLS for c in MACRO_COLS
    )
    assert arms_differ, 'arms are identical — lag shift had no effect on this slice'
    print('✓ Arms differ in macro columns — the A/B is testing a real difference.')

### 3a — In-sample skill per arm

**IS-only fit — interpretation only** (same caveat as nb03): one GBM per arm
trained on the entire stacked slice with no walk-forward. The point is the
*delta between arms*, not the absolute level — IS hit-rate on the training
data is inflated by construction in both arms equally.

In [ ]:
MACRO_ONLY = ['DGS10', 'DFF', 'VIXCLS', 'yield_curve']


def stack_panel(feat_by_sym: dict, cols: list[str]):
    X = np.vstack([feat_by_sym[s][cols].to_numpy() for s in DEMO_SYMBOLS])
    y = np.concatenate([labels_by_sym[s].to_numpy() for s in DEMO_SYMBOLS])
    return X, y


is_stats: dict[str, dict[str, float]] = {}
if REAL_OHLCV_LOADED:
    for arm in ARMS:
        X, y = stack_panel(arms_feats[arm], MACRO_ONLY)
        gbm_is = GBMModel(label_horizon=HORIZON, n_iter=10, n_splits=3, random_state=42)
        gbm_is.fit(X, y)
        pred = gbm_is.predict(X)
        nonzero = y != 0
        is_stats[arm] = {
            'is_hit_rate': float((np.sign(pred) == np.sign(y))[nonzero].mean()),
            'is_corr': float(np.corrcoef(pred, y)[0, 1]),
        }
        print(f'{arm:<9} macro-only IS hit-rate={is_stats[arm]["is_hit_rate"]:.1%}  '
              f'IS corr(pred, label)={is_stats[arm]["is_corr"]:+.3f}')

    hit_delta = is_stats['lagged']['is_hit_rate'] - is_stats['unlagged']['is_hit_rate']
    print(f'\nIS hit-rate delta (lagged − unlagged): {hit_delta:+.2%}')

### 3b — Out-of-sample, walk-forward, per arm

`run_portfolio_backtest` with the GBM preview config (`n_iter=10`,
`n_splits=3`) — identical kwargs for both arms, same `random_state` so the
hyperparameter candidate list is identical and the join timing is the *only*
difference.

In [ ]:
macro_results: dict[str, BacktestResult] = {}
if REAL_OHLCV_LOADED:
    for arm in ARMS:
        macro_results[arm] = run_portfolio_backtest(
            model=GBMModel(label_horizon=HORIZON, n_iter=10, n_splits=3, random_state=7),
            features_by_symbol={s: arms_feats[arm][s][MACRO_ONLY] for s in DEMO_SYMBOLS},
            labels_by_symbol=labels_by_sym,
            prices_by_symbol=gbm_prices_by_sym,
            train_window=504,
            test_window=63,
            step=63,
            label_horizon=HORIZON,
            embargo=3,
        )
        m = macro_results[arm].oos_metrics
        print(f'{arm:<9} macro-only OOS Sharpe={m["sharpe"]:+.3f}  '
              f'MaxDD={m["max_drawdown"]:+.2%}  '
              f'bars={len(macro_results[arm].oos_returns):,}')

In [ ]:
macro_deltas: dict[str, float] = {}
macro_dm = None
if macro_results:
    # Identical inputs ⇒ identical fold structure ⇒ identical OOS index.
    assert macro_results['unlagged'].oos_returns.index.equals(
        macro_results['lagged'].oos_returns.index
    )
    era_macro = tag_regimes(macro_results['lagged'].oos_returns.index, DateRangeDetector())
    print('Regime composition of the OOS panel:')
    print(era_macro.value_counts().rename('bars').to_frame())

    per_arm = {
        arm: compute_regime_metrics(macro_results[arm].oos_returns, era_macro)
        for arm in ARMS
    }
    rows = {}
    for scope in ['aggregate'] + sorted(era_macro.unique()):
        if scope == 'aggregate':
            u = macro_results['unlagged'].oos_metrics['sharpe']
            l = macro_results['lagged'].oos_metrics['sharpe']
        else:
            u = per_arm['unlagged'][scope]['sharpe']
            l = per_arm['lagged'][scope]['sharpe']
        macro_deltas[scope] = l - u
        rows[scope] = {'unlagged': u, 'lagged': l, 'delta (lagged−unlagged)': l - u}
    macro_table = pd.DataFrame.from_dict(rows, orient='index')
    print('\nMacro-only OOS Sharpe by arm:')
    display(macro_table.round(3))

    # Are the arms statistically distinguishable? Two-sided DM on the
    # forecast-error series (H1: the arms' forecast accuracy differs).
    macro_dm = diebold_mariano(
        macro_results['unlagged'].oos_forecast_errors.to_numpy(),
        macro_results['lagged'].oos_forecast_errors.to_numpy(),
        alternative='two-sided',
    )
    print(f'\nDM test, unlagged vs lagged forecast errors (two-sided): '
          f'stat={macro_dm.statistic:+.3f}  p={macro_dm.p_value:.4f}  n={macro_dm.n_obs}')

In [ ]:
if macro_results:
    # Apply the pre-committed interpretation rule, as written in the plan:
    #   branch 1 — the unlagged arm's IS skill *materially degrades* under the
    #              lag → nb03's IS macro dominance is at least partly
    #              mechanical look-ahead;
    #   branch 2 — both arms statistically indistinguishable → join timing is
    #              not the explanation; report "no leak found via publication
    #              timing" for the *attribution* question.
    # Model-level materiality (for re-statement of historical numbers) is a
    # separate question, judged against the pinned thresholds in §5.
    is_hit_drop = is_stats['unlagged']['is_hit_rate'] - is_stats['lagged']['is_hit_rate']
    is_degrades = is_hit_drop > 0
    dm_significant = macro_dm.p_value < 0.05
    probe_material = any(abs(d) > 0.1 for d in macro_deltas.values())

    print('Pre-committed interpretation rule applied to the measured numbers:')
    print(f'  IS hit-rate under lag:   '
          f'{is_stats["unlagged"]["is_hit_rate"]:.1%} → '
          f'{is_stats["lagged"]["is_hit_rate"]:.1%}   (degrades: {is_degrades})')
    print(f'  DM two-sided p:          {macro_dm.p_value:.4f}          '
          f'(significant: {dm_significant})')
    print(f'  OOS Sharpe deltas:       '
          f'{ {k: round(v, 3) for k, v in macro_deltas.items()} }')
    print(f'                           (|Δ| > 0.1 anywhere: {probe_material} '
          f'— feeds the §5 materiality check, not this rule)')

    if is_degrades and dm_significant:
        print('\n→ Branch 1: the unlagged macro-only IS skill degrades under the lag')
        print('  and the arms are statistically distinguishable — nb03\'s IS macro')
        print('  dominance is AT LEAST PARTLY mechanical look-ahead.')
    elif not dm_significant:
        print('\n→ Branch 2: the arms are statistically indistinguishable (and IS')
        print('  skill did NOT degrade under the lag) — join timing is NOT the')
        print('  explanation for the IS-dominant / OOS-absent macro signature.')
        print('  For the attribution question: "no leak found via publication timing."')
    else:
        print('\n→ Mixed outcome: arms statistically distinguishable but IS skill did')
        print('  not degrade — the leak perturbs the model without having been the')
        print('  source of its apparent IS skill.')

### 3c — Probe interpretation (measured 2026-06-12)

The rule's *degradation* branch did **not** fire: macro-only IS hit-rate
**improved** under the lag (56.7% → 59.4%; IS corr 0.464 → 0.523), so the
unlagged arm's apparent in-sample skill was not riding on the 1-day peek.
And the two arms' forecast accuracy is statistically indistinguishable
(DM two-sided p = 0.72). Per the pre-committed rule, **join timing is not
the explanation for nb03's IS-dominant / OOS-absent macro signature** —
that puzzle re-attributes to feature instability or label misspecification
and hands to Milestone 3 as a finding.

Attribution and materiality are different questions, though. The same probe
shows the model's *outputs* are highly mechanically sensitive to the timing
change: covid-regime Sharpe moves by +0.45, and 36% of portfolio-level
forecast signs flip (§4c). Since the DM test cannot tell the arms apart,
this sensitivity reads as **model variance** — the GBM hyperparameter search,
re-run on inputs shifted by one day, lands on materially different fits —
rather than lost predictive information. Materiality for re-statement is
judged on the full-feature A/B against the pinned thresholds in §5.

---
## 4 — Full-feature A/B

The production configuration: the full 17-feature matrix through
`run_portfolio_backtest`, lagged vs unlagged, same GBM preview config and
the same rows as §3. This is the run the pinned materiality thresholds are
evaluated against, because it is the configuration whose historical numbers
(Phase 2.5 / Phase 3) would need re-statement.

Three outputs:
- **(a)** aggregate + per-regime OOS Sharpe delta,
- **(b)** SHAP top-5 stability between arms (nb03's IS-GBM SHAP pattern),
- **(c)** prediction-level diff — fraction of OOS bars where `sign(pred)`
  changes between arms.

In [ ]:
full_results: dict[str, BacktestResult] = {}
if REAL_OHLCV_LOADED:
    for arm in ARMS:
        full_results[arm] = run_portfolio_backtest(
            model=GBMModel(label_horizon=HORIZON, n_iter=10, n_splits=3, random_state=7),
            features_by_symbol=arms_feats[arm],
            labels_by_symbol=labels_by_sym,
            prices_by_symbol=gbm_prices_by_sym,
            train_window=504,
            test_window=63,
            step=63,
            label_horizon=HORIZON,
            embargo=3,
        )
        m = full_results[arm].oos_metrics
        print(f'{arm:<9} full-feature OOS Sharpe={m["sharpe"]:+.3f}  '
              f'MaxDD={m["max_drawdown"]:+.2%}  '
              f'bars={len(full_results[arm].oos_returns):,}')

### 4a — Aggregate + per-regime OOS Sharpe delta

In [ ]:
full_deltas: dict[str, float] = {}
era_full = None
if full_results:
    assert full_results['unlagged'].oos_returns.index.equals(
        full_results['lagged'].oos_returns.index
    )
    era_full = tag_regimes(full_results['lagged'].oos_returns.index, DateRangeDetector())

    for arm in ARMS:
        print(f'\n{arm} — per-regime summary (regime_summary_table):')
        display(regime_summary_table(full_results[arm], era_full).round(3))

    per_arm_full = {
        arm: compute_regime_metrics(full_results[arm].oos_returns, era_full)
        for arm in ARMS
    }
    rows = {}
    for scope in ['aggregate'] + sorted(era_full.unique()):
        if scope == 'aggregate':
            u = full_results['unlagged'].oos_metrics['sharpe']
            l = full_results['lagged'].oos_metrics['sharpe']
        else:
            u = per_arm_full['unlagged'][scope]['sharpe']
            l = per_arm_full['lagged'][scope]['sharpe']
        full_deltas[scope] = l - u
        rows[scope] = {'unlagged': u, 'lagged': l, 'delta (lagged−unlagged)': l - u}
    full_table = pd.DataFrame.from_dict(rows, orient='index')
    print('\nFull-feature OOS Sharpe by arm:')
    display(full_table.round(3))

### 4b — SHAP top-5 stability between arms

nb03's pattern: one **IS-only** GBM per arm on the stacked slice (preview
`n_iter=10` to keep it cheap), SHAP via native
`booster.predict(pred_contribs=True)` — no `shap` library required. If the
unlagged macro dominance was a timing artifact, the lagged arm's ranking
should demote the macro family; if the rankings are essentially identical,
the SHAP story is insensitive to the leak.

In [ ]:
shap_imp: dict[str, pd.Series] = {}
if REAL_OHLCV_LOADED:
    for arm in ARMS:
        X, y = stack_panel(arms_feats[arm], FULL_COLS)
        gbm_arm = GBMModel(label_horizon=HORIZON, n_iter=10, n_splits=3, random_state=42)
        gbm_arm.fit(X, y)
        booster = gbm_arm._model.get_booster()
        dm_mat = xgb.DMatrix(X, feature_names=FULL_COLS)
        contribs = booster.predict(dm_mat, pred_contribs=True)[:, :-1]
        shap_imp[arm] = pd.Series(
            np.abs(contribs).mean(axis=0), index=FULL_COLS
        ).sort_values(ascending=False)

    top5 = pd.DataFrame({
        'unlagged_top5': shap_imp['unlagged'].head(5).index,
        'unlagged_mean|SHAP|': shap_imp['unlagged'].head(5).to_numpy(),
        'lagged_top5': shap_imp['lagged'].head(5).index,
        'lagged_mean|SHAP|': shap_imp['lagged'].head(5).to_numpy(),
    }, index=pd.RangeIndex(1, 6, name='rank'))
    display(top5.round(5))

    overlap = set(shap_imp['unlagged'].head(5).index) & set(shap_imp['lagged'].head(5).index)
    rank_u = shap_imp['unlagged'].rank(ascending=False)
    rank_l = shap_imp['lagged'].rank(ascending=False)
    rank_corr = float(rank_u.corr(rank_l, method='spearman'))
    macro_in_top5 = {
        arm: sorted(set(shap_imp[arm].head(5).index) & set(MACRO_ONLY)) for arm in ARMS
    }
    print(f'Top-5 overlap between arms: {len(overlap)}/5  ({sorted(overlap)})')
    print(f'Spearman rank correlation across all {len(FULL_COLS)} features: {rank_corr:+.3f}')
    for arm in ARMS:
        print(f'Macro features in {arm} top-5: {macro_in_top5[arm]}')

### 4c — Prediction-level diff: sign-flip fraction

**Derivation choice (documented per the plan).** `BacktestResult` does not
store per-symbol raw predictions. But `run_portfolio_backtest` computes, per
OOS bar, `oos_forecast_errors = mean(label) − mean(pred)` where both means
are cross-sectional over the symbols active at that bar — and §3 asserted
that every symbol in this slice shares one calendar, so the cross-sectional
mean label is exactly reconstructable from `labels_by_sym`. The
**portfolio-level forecast** therefore recovers exactly as
`pred = mean(label) − error`, with zero extra model fits. This is the
cheapest correct path; the alternative (a per-fold re-prediction loop) would
re-fit ~20 GBM folds per arm for the same information at portfolio level.

The measured quantity is the sign of the portfolio-level (cross-sectional
mean) forecast. Per-symbol trade signals are `sign(pred_sym)`, which this
aggregates; a portfolio-level sign flip requires at least one symbol-level
prediction to have moved, so it is a *conservative* (lower-bound) reading of
symbol-level sensitivity.

In [ ]:
sign_flip_frac = pred_changed_frac = None
if full_results:
    label_mean = pd.concat(
        [labels_by_sym[s] for s in DEMO_SYMBOLS], axis=1
    ).mean(axis=1)

    pred_mean = {}
    for arm in ARMS:
        err = full_results[arm].oos_forecast_errors
        pred_mean[arm] = label_mean.loc[err.index] - err

    common = pred_mean['unlagged'].index  # identical across arms (asserted in §4a)
    flips = np.sign(pred_mean['unlagged']) != np.sign(pred_mean['lagged'])
    changed = pred_mean['unlagged'] != pred_mean['lagged']
    sign_flip_frac = float(flips.mean())
    pred_changed_frac = float(changed.mean())

    print(f'OOS bars compared:                       {len(common):,}')
    print(f'Bars where the forecast value changed:   {pred_changed_frac:.1%}')
    print(f'Bars where sign(pred) flipped:           {sign_flip_frac:.2%}')
    print(f'Pinned materiality threshold:            5.00%')

    # Same diagnostic for the macro-only probe (not gate-bearing, but the
    # most leak-sensitive configuration).
    pred_mean_macro = {
        arm: label_mean.loc[macro_results[arm].oos_forecast_errors.index]
        - macro_results[arm].oos_forecast_errors
        for arm in ARMS
    }
    macro_flips = np.sign(pred_mean_macro['unlagged']) != np.sign(pred_mean_macro['lagged'])
    print(f'\n(macro-only probe sign-flip fraction:    {float(macro_flips.mean()):.2%})')

---
## 5 — Verdict

The pinned thresholds (pre-committed in the plan before any result existed):
**material** if sign-flip fraction > 5% of OOS bars **or** |ΔSharpe| > 0.1 in
aggregate or in any era regime — evaluated on the full-feature A/B, the
configuration whose historical numbers would need re-statement. The leak is
*confirmed* mechanically by §2 regardless (the feature values the model saw
were not knowable at the close); the classification below is about
*materiality* at the model level.

In [ ]:
if full_results:
    SIGN_FLIP_THRESHOLD = 0.05
    SHARPE_DELTA_THRESHOLD = 0.10

    sharpe_trips = {k: abs(v) > SHARPE_DELTA_THRESHOLD for k, v in full_deltas.items()}
    sign_flip_trips = sign_flip_frac > SIGN_FLIP_THRESHOLD
    material = sign_flip_trips or any(sharpe_trips.values())
    mechanically_sensitive = pred_changed_frac > 0 or any(v != 0 for v in full_deltas.values())

    print('Pinned thresholds vs measured values (full-feature A/B):')
    print(f'  sign-flip fraction   {sign_flip_frac:7.2%}  '
          f'(threshold > {SIGN_FLIP_THRESHOLD:.0%})  → trips: {sign_flip_trips}')
    for scope, delta in full_deltas.items():
        print(f'  |ΔSharpe| {scope:<12} {abs(delta):7.3f}  '
              f'(threshold > {SHARPE_DELTA_THRESHOLD:.1f})  → trips: {sharpe_trips[scope]}')

    if material:
        verdict = 'LEAK CONFIRMED + MATERIAL'
    elif mechanically_sensitive:
        verdict = 'LEAK CONFIRMED + IMMATERIAL'
    else:
        verdict = 'NO MECHANICAL SENSITIVITY'

    print(f'\n{"=" * 60}')
    print(f'VERDICT: {verdict}')
    print(f'{"=" * 60}')
    print('\nThe lagged join stays the build_features default REGARDLESS of this')
    print('verdict — the adoption decision was pre-committed on correctness grounds.')

### Measured verdict (2026-06-12, slice-level)

**LEAK CONFIRMED + MATERIAL** — every pinned threshold trips on the
full-feature A/B:

| Quantity | Measured | Pinned threshold |
|---|---|---|
| sign-flip fraction of OOS bars | **23.27%** | > 5% |
| &#124;ΔSharpe&#124; aggregate | **0.265** (−0.540 → −0.806) | > 0.1 |
| &#124;ΔSharpe&#124; `covid` | **0.253** (+1.614 → +1.867) | > 0.1 |
| &#124;ΔSharpe&#124; `rate_cycle` | **0.377** (−0.905 → −1.282) | > 0.1 |

Two findings travel together and must not be conflated:

1. **Re-statement (material).** All pre-fix numbers — Phase 2.5, Phase 3,
   nb02–nb06 — were computed under the unlagged join and are not trustworthy
   at the ±0.1 Sharpe granularity. The deltas cut *both* directions by regime
   (lagged is better in `covid`, worse in `rate_cycle` and aggregate on this
   slice), so this is sensitivity, not a uniform haircut.
2. **Attribution (negative finding).** The leak does **not** explain nb03's
   IS macro dominance: macro-only IS skill *survives* (indeed improves) under
   the lag and the arms are statistically indistinguishable (DM p = 0.72,
   §3c). SHAP rankings are likewise stable between arms (top-5 overlap 4/5,
   Spearman ρ = +0.93 — §4b). The IS-dominant / OOS-absent puzzle
   re-attributes to feature instability or label misspecification.

**What's next.**

- **Milestone 3 (regime-aware features)** inherits both findings: its
  per-feature ablation baselines run on the corrected join, and the macro
  IS/OOS puzzle is now a feature/label question, not a join-timing question.
- **Milestone 6 (exit gate)** runs the full 33-symbol × 23-year panel on the
  corrected join; those numbers supersede every pre-fix result. nb02/nb04 are
  deliberately *not* re-run here.
- The lagged join is already the `build_features` default;
  `fred_publication_lags=None` remains only as the escape hatch for
  reproducing pre-fix historical results.
- The `_load_fred_wide` session-timezone CAST artifact found during this
  audit (§2 side-finding) is fixed and regression-tested.

See [`docs/concepts/fred-publication-lag.md`](../docs/concepts/fred-publication-lag.md)
for the pinned lag evidence and the update protocol.